Script for Calculating AI\-Human Correlation \(Pearson\-R\)

In [1]:
def pearsonr_correlation(human_df, gemini_df):

    # 1. Merge the two dataframes on the 'stimulus' column
    # Added suffixes to tell which column is which (e.g., trustworthy_human vs trustworthy_gemini)
    merged_df = pd.merge(human_df, gemini_df, on='stimulus', suffixes=('_human', '_gemini'))

    # The list of baseline attributes we want to loop through
    attributes = [col for col in gemini_df.columns if col != 'stimulus']

    # A dictionary to store our final correlation results
    correlation_results = {}

    for attr in attributes:
        # Construct the exact column names we generated during the merge
        human_col = f"{attr}_human"
        gemini_col = f"{attr}_gemini"

        # Calculate the correlation between the human and gemini columns
        current_r = merged_df[human_col].corr(merged_df[gemini_col])

        # Store the result in dictionary
        correlation_results[attr] = current_r

    # Convert to a DataFrame for easy plotting later
    corr_df = pd.DataFrame(list(correlation_results.items()), columns=['Attribute', 'Gemini_Correlation'])
    corr_df = corr_df.sort_values(by="Gemini_Correlation", ascending=False)

    return corr_df

Script for Calculating Human Interrater Reliability \(R^2\):

Used to emulate Fig\. 2 in PNAS study, which plots average cross\-validated model performance vs\. intersubject reliability, to see how model performs compared to other humans\. As in the PNAS study, we compare the average squared pearson correlation \(r^2\) of the averages of two random split\-halves across 100 random splits to determine interrater reliability\.

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr


def calculate_human_reliability(df_human: pd.DataFrame, attribute_name: str, n_splits: int = 100) -> float:
    """
    Calculates the split-half reliability (R^2) for a single facial attribute accross 100 random splits.
    """

    # 1. Filter the data
    df_attr = df_human[df_human['attribute'] == attribute_name]

    split_r2s = []
    
    for split in range(n_splits):
        group_a = []
        group_b = []

        # Group by stimulus to shuffle ratings locally per image
        for stim, group in df_attr.groupby('stimulus'):
            ratings = group['rating'].values.copy() # Make a copy to avoid in-place corruption

            # Shuffle the ratings
            np.random.shuffle(ratings)

            len_ratings = len(ratings)
            mid = len_ratings // 2

            if mid == 0:
                continue

            # Extract the split-halves and calculate their means
            mean_a = np.mean(ratings[:mid])
            mean_b = np.mean(ratings[mid:])

            group_a.append({'stimulus': stim, 'mean_rating_a': mean_a})
            group_b.append({'stimulus': stim, 'mean_rating_b': mean_b})

        # Convert lists to temporary DataFrames
        df_a = pd.DataFrame(group_a)
        df_b = pd.DataFrame(group_b)

        if df_a.empty or df_b.empty:
            continue
            
        # Merge on stimulus to ensure matching row alignment
        merged_splits = pd.merge(df_a, df_b, on='stimulus')

        # Calculate Pearson correlation (returns tuple: (r_value, p_value))
        r_split, _ = pearsonr(merged_splits['mean_rating_a'], merged_splits['mean_rating_b'])

        # Convert r to R^2 and store it in split_r2s
        split_r2s.append(r_split ** 2)

    # Return the mean of all successful splits
    return np.mean(split_r2s) if split_r2s else np.nan

Script for Computing AI performance \(R^2\)

In [3]:
import pandas as pd

def build_evaluation_dataset(df_human_means, df_human_raw, df_ai):
    """
    Loops through overlapping attributes, calculates AI and Human R^2 metrics,
    and returns a sorted DataFrame ready for visualization.
    """

    # Find matching attributes present in both datasets
    ai_attributes = [col for col in df_ai.columns if col != 'stimulus']
    human_attributes = df_human_raw['attribute'].unique()
    attributes_to_plot = list(set(ai_attributes).intersection(set(human_attributes)))
    
    results = []
    
    ai_correlations = pearsonr_correlation(df_human_means, df_ai)
    
    # Turn the 'Attribute' column into the row indices
    ai_lookup = ai_correlations.set_index('Attribute')

    for attr in attributes_to_plot:
        print(f"Calculating metrics for: {attr}...")
        
        # 1. Compute Human Intersubject Reliability 
        human_r2 = calculate_human_reliability(df_human_raw, attr, n_splits=100)
        
        # 2. Compute AI Performance R^2
        ai_r = ai_lookup.loc[attr, 'Gemini_Correlation']
        ai_r2 = ai_r ** 2
        
        # Append to our structured results list
        results.append({
            'attribute': attr,
            'human_reliability_r2': human_r2,
            'ai_performance_r2': ai_r2
        })
        
    # Turn results into a DataFrame and sort by human ceiling (mimicking PNAS)
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by='human_reliability_r2', ascending=True).reset_index(drop=True)
    return df_results


Plotting Functions:

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

def plot_model_comparison(df_results: pd.DataFrame):
    """
    Generates a horizontal layered bar chart comparing AI performance against human intersubject reliability boundaries.
    """

    # 1. Establish canvas dimensions based on how many attributes we are plotting (0.4 inches per attribute)
    fig, ax = plt.subplots(figsize=(10, len(df_results) * 0.4 + 2), dpi=150)
    y_positions = np.arange(len(df_results))

    # --- BACK LAYER (Human reliability ceiling) ---
    ax.barh(y_positions, df_results['human_reliability_r2'], color='#fcdcdb', 
            edgecolor='none', label='Model Shortfall', height=0.5)
    
    # --- MIDDLE LAYER (AI Model Performance) ---
    ax.barh(y_positions, df_results['ai_performance_r2'], color='black', 
            edgecolor='none', label='Model', height=0.5)
    
    # --- FRONT LAYER (Split-Half Red Ticks & Integer Labels) ---
    for idx, row in df_results.iterrows():
        # Draw the red tick mark at the human reliability ceiling
        ax.plot([row['human_reliability_r2'], row['human_reliability_r2']], 
                [idx - 0.25, idx + 0.25], color='#d62728', linewidth=2.5, zorder=3)

        # Convert raw floating points to rounded percentages (e.g. 0.64 -> 64)
        model_pct = int(round(row['ai_performance_r2'] * 100))
        human_pct = int(round(row['human_reliability_r2'] * 100))
        
        # Place the numerical strings cleanly to the right of their respective targets
        # The '+ 0.01' is a visual buffer padding to keep the text from overlapping the bars
        ax.text(row['ai_performance_r2'] + 0.01, idx, f"{model_pct}", 
                va='center', ha='left', fontsize=9, color='black')
        ax.text(row['human_reliability_r2'] + 0.01, idx, f"{human_pct}", 
                va='center', ha='left', fontsize=9, color='#d62728', fontweight='bold')

    # --- CHART STYLING & CLEANUP ---
    ax.set_yticks(y_positions)
    
    # Automatically clean up formatting by swapping out underscores for spaces in the names
    ax.set_yticklabels([attr.replace('_', ' ') for attr in df_results['attribute']], fontsize=10)
    
    # Use LaTeX notation in the label ($R^2$)
    ax.set_xlabel('Out-of-sample $R^2$', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1.05)
    
    # Clean up margins by turning off the non-essential top and right borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    
    # Build a highly specific custom legend layout
    legend_elements = [
        Line2D([0], [0], color='#d62728', lw=2.5, label='Split-Half Ceiling'),
        Line2D([0], [0], color='black', lw=6, label='Gemini Model'),
        Line2D([0], [0], color='#fcdcdb', lw=6, label='Model Shortfall')
    ]

    ax.legend(handles=legend_elements, loc='lower right', frameon=False, bbox_to_anchor=(1, 0.1))
    
    plt.title('Average Cross-Validated Model Performance compared to Intersubject Reliability', 
              fontsize=12, pad=20, fontstyle='italic')
    plt.tight_layout()
    plt.show()



Running the 

In [5]:
# Load predictions 
human_df_means = pd.read_csv("/work/20260629-213612/omi-main/attribute_means.csv")
gemini_df = pd.read_csv("/work/gemini_predictions.csv")
human_df_raw = pd.read_csv("/work/20260629-213612/omi-main-20260701-222336/attribute_ratings.csv") 

results_df = build_evaluation_dataset(human_df_means, human_df_raw, gemini_df)
print(results_df)
plot_model_comparison(results_df)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=db20d035-0f07-44b3-9147-5cdb9d6d1fdd' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>